# Tutorial — EnergyScope Pathway Model, with materials

This notebook shows how to run the EnergyScope transition-pathway model **with critical-materials
tracking** programmatically using `run_pathway(..., materials=True, ...)` (or the simpler
`run_materials_scenario()` wrapper) from `shared.utils`, access the results, and generate plots.

It follows the same structure as the plain-pathway tutorial (`projects/pathway/docs/tutorial_pathway.ipynb`)
-- read that one first if you haven't already, everything there (basic run, plotting, saving, GWP budget,
myopic mode, ...) still applies unchanged. This notebook only covers what's *added* by `materials=True`.

Make sure you are running this notebook from `projects/critical_materials/` (or that this directory and
the repo root are both on your Python path).

## 1. Import

In [1]:
import sys
sys.path.insert(0, '../..')  # repo root, for `shared.utils`

from shared.utils import run_materials_scenario, run_pathway

## 2. Basic run

`run_materials_scenario()` is a beginner-friendly wrapper around `run_pathway(materials=True, ...)` --
it picks a sensible combination of the materials-specific flags for you (see §6). `mode='free'` means
no cost signal: `Recycled_material` is forced to the technical recycling-rate ceiling.

In [2]:
results = run_materials_scenario('test_scenario', mode='free')

[run_pathway] Window 1/1 done in 468.5s
[run_pathway] Total time: 468.5s
Plotting to: /Users/Paolo/Documents/PdM_code/EnergyScope-Quebec/projects/critical_materials/../../projects/critical_materials/out/test_scenario/graphs
  Plotting [215]: 16_Sankey_2050.html                                    [WARN] GWP 2040: Year_balance=37816.0 kt, allocated=31645.7 kt (diff=-6170.3 kt)
[WARN] GWP 2045: Year_balance=37816.0 kt, allocated=28222.3 kt (diff=-9593.7 kt)
  Plotting [236]: 18_Elec_monthly_2050.html                                 Dashboard: /Users/Paolo/Documents/PdM_code/EnergyScope-Quebec/projects/critical_materials/../../projects/critical_materials/out/test_scenario/graphs/index.html

Done — 236 charts saved to /Users/Paolo/Documents/PdM_code/EnergyScope-Quebec/projects/critical_materials/../../projects/critical_materials/out/test_scenario/graphs
  Plotting [358]: 24_Material_mult_h2_prod.html                            Dashboard: /Users/Paolo/Documents/PdM_code/EnergyScope-Quebec/pr

## 3. Exploring results

Same dict of ~30 DataFrames as the plain pathway model, plus material-specific keys.

In [3]:
# Material-specific result keys
[k for k in results if 'material' in k.lower() or 'recycl' in k.lower()]

['Material_content_year',
 'Decommissioned_material',
 'Recycled_material',
 'Recycled_material_by_process',
 'Disposed_material',
 'Recycling_benefit',
 'Recycling_shortfall',
 'C_material',
 'C_material_recycling_tech',
 'Material_content_cumulative',
 'Recycled_material_cumulative',
 'Recycling_benefit_cumulative']

In [4]:
# Annual material demand per (year, technology, material) [t/year]
results['Material_content_year'].head()

Material_content_year
Years     Technologies Materials                       
YEAR_2020 AFC          Ag                           0.0
                       Al                           0.0
                       B                            0.0
                       Cd                           0.0
                       Co                           0.0

In [5]:
# Annual amount actually recycled, by material, summed across technologies [t/year]
rec = results['Recycled_material']['Recycled_material']
rec[rec.abs() > 1e-9].groupby('Materials').sum().sort_values(ascending=False)

Materials
Fe          1.591029e+06
Al          1.553820e+05
Cu          2.819245e+04
Li          1.609073e+04
Cr          6.511741e+03
Ni          5.250370e+03
Mn          5.214406e+03
Zn          3.609332e+03
Glass       9.542685e+02
Co          8.754680e+02
Pb          5.507273e+02
Polymers    4.691806e+02
Nd          3.783193e+02
Mo          9.392584e+01
Dy          4.457391e+01
Ag          3.473870e+01
Pr          4.036714e+00
Mg          2.021899e+00
Tb          5.811486e-01
B           5.706593e-01
Te          6.967852e-03
Cd          6.589164e-03
Pt          5.457651e-03
In          1.860430e-03
Si          1.211036e-03
Ga          5.952539e-04
Se          2.079638e-04
Ge          1.619847e-04
Name: Recycled_material, dtype: float64

In [6]:
# Material demand net of recycling ('economic cost avoided by recycling') [M$/year]
results['Recycling_benefit_cumulative'] if results.get('Recycling_benefit_cumulative') is not None \
    else 'Only present when materials_recycling_cost=True (mode=\'real_cost\')'

Recycling_benefit_cumulative
Years     Technologies  Materials                              
YEAR_2020 AFC           Ag                                  0.0
                        Al                                  0.0
                        B                                   0.0
                        Cd                                  0.0
                        Co                                  0.0
...                                                         ...
YEAR_2050 WOOD_METHANOL V                                   0.0
                        W                                   0.0
                        Y                                   0.0
                        Zn                                  0.0
                        Zr                                  0.0

[202909 rows x 1 columns]

## 4. Plotting

`build_dashboard=True` (the default when `materials=True`) automatically generates
`out/<case_study>/graphs/index.html` -- a superset of the plain-pathway dashboard, with extra pages for
material demand, decommissioning, and recycling by material/technology.

In [ ]:
results = run_materials_scenario('test_scenario', mode='free', open_dashboard=True)

## 5. Saving results to disk

Same `save_pkl`/`skip_if_exists` as the plain pathway model -- `materials=True` additionally writes
`_Materials_Results.pkl` alongside `_Results.pkl`.

In [ ]:
results = run_materials_scenario(
    'my_first_materials_run',
    mode='free',
    description='Baseline materials run',
)  # skip_if_exists=True to reload from disk instead of re-solving, same as the plain model

## 6. The materials pipelines -- where the data comes from

Nothing in `ampl_files/*.dat` is written by hand: `Material_intensity.dat` and `Material_recycling.dat`
are **auto-generated** from the Excel workbooks in `excel_files/`. If you edit an Excel sheet, rerun the
matching pipeline before your next `run_materials_scenario()`/`run_pathway()` call, or the change won't
take effect.

| Pipeline | Source Excel | Output | Script |
|---|---|---|---|
| `mi_pipeline` | `Material_intensities_energyscope.xlsx` | `Material_intensity.dat` | `run_build_mi.py` |
| `rr_pipeline` | `Recycling_rates.xlsx` | `Material_recycling.dat` | `run_build_rr.py` |
| `rt_pipeline` | `Recycling_rates.xlsx` | `Material_recycling_process.dat` | `run_build_rt.py` (only needed for `materials_recycling_process=True`, see §7) |

In [ ]:
from run_build_mi import main as build_mi
from run_build_rr import main as build_rr

build_mi(write_xlsx=False)  # write_xlsx=True (default) also regenerates the audit workbook, ~5 min instead of seconds
build_rr()

## 7. Recycling approaches

`materials_recycling=True` is required for any recycling at all (otherwise `recycling_rate` stays 0
everywhere and everything is disposed). On top of that, three independent flags control *how much* gets
recycled:

- `materials_recycling_cost` (default `True`): real recycling/disposal costs drive the optimizer's
  choice. Set `False` for no cost signal at all -- in that case `Recycled_material` is otherwise
  solver-indeterminate unless you also set...
- `force_max_recycling` (default `False`): forces `Recycled_material` to exactly the technical ceiling
  (`recycling_rate`). This is what `mode='free'` uses in `run_materials_scenario()`.
- `follow_objective` (default `False`): forces a **minimum** equal to `recycling_objective_share` (a
  policy/regulatory target, e.g. an EU Critical Raw Materials Act quota) instead of just the technical
  ceiling.

`run_materials_scenario(mode='real_cost')` is the shortcut for real costs, no forcing.

In [ ]:
results_free = run_materials_scenario('approach_free', mode='free')
results_cost = run_materials_scenario('approach_real_cost', mode='real_cost')
results_objective = run_pathway('approach_objective', materials=True, materials_recycling=True,
                                 follow_objective=True, description='Follows recycling_objective_share')

## 8. Material production limits

`mat_limit=True` (or the underlying `materials_limit=True`) applies manual annual caps from
`Material_limits.dat` -- e.g. a Quebec-allocated share of world neodymium production, so the model can't
assume unlimited access to a genuinely scarce material.

In [ ]:
results_mat_limit = run_materials_scenario('with_mat_limit', mode='free', mat_limit=True)